## Importing data:

In [6]:
import pandas as pd

train_path = '/kaggle/input/competitions/nlp-getting-started/train.csv'
test_path = '/kaggle/input/competitions/nlp-getting-started/test.csv'

train_df = pd.read_csv(train_path)
print('Training set: ',train_df.shape)
display(train_df.head(3))

test_df = pd.read_csv(test_path)
print('Test set: ',test_df.shape)
display(test_df.head(3))

Training set:  (7613, 5)


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1


Test set:  (3263, 4)


,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."


In [7]:
print(f"Total samples in the training set are: {len(train_df)}")

Total samples in the training set are: 7613


In [8]:
target = 'target'

#we'll be using bag-of-word model so we will only require text column and the target
X = train_df['text'].values
y = train_df[target].values

## Preparing the data:
We'll split the train set into train/val set using **train_test_split()**, we'll have about 20% of training data in validation split.

In [9]:
from sklearn.model_selection import train_test_split

X_train,X_val,y_train,y_val = train_test_split(X,y, 
                                               test_size=0.2,
                                               stratify=y, 
                                               random_state=42)

### Creating tf.dataset objects:

In [10]:
import tensorflow as tf

train_ds = tf.data.Dataset.from_tensor_slices((X_train,y_train))
train_ds = train_ds.batch(16)

val_ds = tf.data.Dataset.from_tensor_slices((X_val,y_val))
val_ds = val_ds.batch(16)

2026-03-16 16:30:42.723286: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


### Preparing int sequences for model:

In [11]:
from tensorflow.keras.layers import TextVectorization

max_tokens = 10000

vectorizer = TextVectorization(
    max_tokens = max_tokens,
    output_mode = 'int'
)

vectorizer.adapt(X_train)

In [12]:
train_ds_int = train_ds.map(lambda x, y: (vectorizer(x),y),
                            num_parallel_calls=4)

val_ds_int = val_ds.map(lambda x, y: (vectorizer(x),y), 
                            num_parallel_calls=4)

## Sequence-Model OneHotEncoded vectors based:
The simplest way to convert our integer sequences to vector
sequences is to one-hot encode the integers (each dimension would represent one
possible term in the vocabulary). On top of these one-hot vectors, we’ll add a simple
bidirectional LSTM.

In [19]:
from tensorflow import keras
from tensorflow.keras import layers

inputs = keras.Input(shape=(None,), dtype='int64') #our input is sequence of integers
embedded = layers.Lambda(
    lambda x: tf.one_hot(x, depth=max_tokens),
    output_shape=(None, max_tokens)
) (inputs) #encodes the integers into binary 10000-D vector
x = layers.Bidirectional(layers.LSTM(32)) (embedded)
x = layers.Dropout(0.5) (x)
outputs = layers.Dense(1, activation='sigmoid') (x) #output/classification layer

model = keras.Model(inputs, outputs)
model.compile(optimizer='rmsprop',
             loss='binary_crossentropy',
             metrics=['accuracy'])
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_1 (Lambda)               │ (None, None, 10000)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 64)             │     2,568,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,568,513 (9.80 MB)

 Trainable params: 2,568,513 (9.80 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
callbacks = [
    keras.callbacks.ModelCheckpoint('one_hot_lstm.keras', save_best_only=True)
]

model.fit(train_ds_int,
         validation_data=val_ds_int,
         callbacks=callbacks,
         epochs=10)

Epoch 1/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 34s 85ms/step - accuracy: 0.6122 - loss: 0.6534 - val_accuracy: 0.7754 - val_loss: 0.4928
Epoch 2/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 31s 81ms/step - accuracy: 0.7931 - loss: 0.4644 - val_accuracy: 0.8037 - val_loss: 0.4578
Epoch 3/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 31s 82ms/step - accuracy: 0.8438 - loss: 0.3843 - val_accuracy: 0.7919 - val_loss: 0.4799
Epoch 4/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 31s 83ms/step - accuracy: 0.8656 - loss: 0.3333 - val_accuracy: 0.8043 - val_loss: 0.4845
Epoch 5/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 31s 83ms/step - accuracy: 0.8868 - loss: 0.2978 - val_accuracy: 0.7958 - val_loss: 0.4963
Epoch 6/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 30s 80ms/step - accuracy: 0.9020 - loss: 0.2783 - val_accuracy: 0.7938 - val_loss: 0.5108
Epoch 7/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 42s 82ms/step - accuracy: 0.9125 - loss: 0.2485 - val_accuracy: 0.7945 - val_loss: 0.5513
Epoch 8/10
381/381 ━━━━━━━━━━━━━━━━━━━━ 33s 87ms/step - accuracy: 0.9184 - loss: 0.2352 - 

This model trains very slowly, especially compared to the light
weight model from teh bag-of-words approach. This is because our inputs are quite large. This model is bound to perform badly on test data since our valiation score never reached as high as teh bag-of-words models.

Clearly, using one-hot encoding to turn words into vectors, which was the simplest
thing we could do, wasn’t a great idea. There’s a better way: **word embeddings**. 